In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
PROJECT_DIR = Path(__file__).resolve().parent.parent
DATA_PATH = PROJECT_DIR / "data" / "Sample - Superstore.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"

In [ ]:
def main():
    df = pd.read_csv(DATA_PATH, parse_dates=["Order_Date"])
    OUTPUT_DIR.mkdir(exist_ok=True)

    print("TOTAL ORDERS IN DATASET:", len(df))
    print("ORDER STATUS COUNTS:")
    print(df["Order_Status"].value_counts().to_string())

    # Provisional assumption: only delivered orders count as completed purchases.
    delivered = df[df["Order_Status"] == "Delivered"].copy()

    if delivered.empty:
        raise ValueError("No delivered orders found for RFM analysis.")

    reference_date = df["Order_Date"].max().normalize() + pd.Timedelta(days=1)

    rfm = (
        delivered.groupby("Customer_ID")
        .agg(
            Recency=("Order_Date", lambda dates: (reference_date - dates.max()).days),
            Frequency=("Order_ID", "nunique"),
            Monetary=("Net_Sales", "sum"),
        )
    )

    # Rank-based scoring is more robust when many customers have similar values.
    rfm["R_Score"] = pd.qcut(
        rfm["Recency"].rank(method="first"),
        4,
        labels=[4, 3, 2, 1],
    ).astype(int)

    rfm["F_Score"] = pd.qcut(
        rfm["Frequency"].rank(method="first"),
        4,
        labels=[1, 2, 3, 4],
    ).astype(int)

    rfm["M_Score"] = pd.qcut(
        rfm["Monetary"].rank(method="first"),
        4,
        labels=[1, 2, 3, 4],
    ).astype(int)

    rfm["RFM_Score"] = (
        rfm["R_Score"].astype(str)
        + rfm["F_Score"].astype(str)
        + rfm["M_Score"].astype(str)
    )

    def assign_segment(row):
        if row["R_Score"] >= 3 and row["F_Score"] >= 3 and row["M_Score"] >= 3:
            return "Best Customers"
        elif row["R_Score"] >= 3 and row["F_Score"] >= 2:
            return "Loyal / Active"
        elif row["R_Score"] <= 2 and row["F_Score"] >= 3:
            return "At Risk"
        elif row["R_Score"] >= 3 and row["F_Score"] == 1:
            return "Recent Customers"
        else:
            return "Needs Attention"

    rfm["Customer_Segment"] = rfm.apply(assign_segment, axis=1)
    rfm = rfm.sort_values(
        ["RFM_Score", "Monetary"],
        ascending=[False, False],
    )

    print("\nRFM CUSTOMER SEGMENTATION")
    print("Reference date:", reference_date.date())
    print("Delivered orders used:", len(delivered))
    print("Customers segmented:", len(rfm))
    print("\nCUSTOMER SEGMENT COUNTS:")
    print(rfm["Customer_Segment"].value_counts().to_string())
    print("\nTOP 10 CUSTOMERS BY MONETARY VALUE:")
    print(rfm.head(10).round(2).to_string())

    output_path = OUTPUT_DIR / "customer_rfm_segmentation.csv"
    rfm.to_csv(output_path, index_label="Customer_ID")

    print("\nRFM results saved to:", output_path)
    print(
        "Note: RFM uses delivered orders and Net_Sales as a proxy for "
        "customer value. Validate returns and accounting rules before "
        "using these figures for real business decisions."
    )

In [ ]:
if __name__ == "__main__":
    main()